Load dataset

In [ ]:
import numpy as np
import pandas as pd
data = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
labels = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols=(['species']), dtype=str)

Introducing missing values

In [ ]:
import os
os.chdir("C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\DadosAusentes")

def binary_sampler(p, rows, cols):
    unif_random_matrix = np.random.uniform(0.0, 1.0, size=[rows, cols])
    binary_random_matrix = (unif_random_matrix < p).astype(np.float32)
    return binary_random_matrix

# Introduce missing data
no, dim = data.shape
miss_rate = [0.05,0.1,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50]


for item in miss_rate:
    data_m = binary_sampler(1-item, no, dim)
    miss_data_x = data.copy()
    miss_data_x[data_m == 0] = np.nan
    #miss_data_x.fillna("?", inplace = True)   
    miss_data_x = pd.concat([miss_data_x, labels], axis=1)
    miss_data_x.to_csv('Iris'+ str(item) + '.csv',  sep=';', encoding="latin-1", index = False)

os.chdir("C:\Anaconda3\Scripts\TestesPesquisa")

RMSE Function

In [ ]:
def rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())

Mean Imputation

In [ ]:
for item in miss_rate:
    data = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\DadosAusentes\Iris'+ str(item) + '.csv', sep=';', error_bad_lines=False, encoding="latin-1")
    rmse_val = 0
    i = 0
    while (i < 10):
  
        data_mean_imput = data.fillna(data.mean())
        data_mean_imput.to_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Mean\Iris'+ str(item) + '_Mean.csv',  sep=';', encoding="latin-1", index = False)        
        
        predict = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Mean\Iris'+ str(item) + '_Mean.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
        real = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
        
        rmse_val  = rmse_val + rmse(np.array(predict), np.array(real))
        i = i + 1
    
    print("Root Mean Square Error (" + str(item) + "% missing values): " + str(rmse_val/10))
    

kNN Imputation

In [ ]:
from missingpy import KNNImputer
import warnings
warnings.filterwarnings("ignore")

imputer = KNNImputer()

for item in miss_rate:
    data = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\DadosAusentes\Iris'+ str(item) + '.csv', sep=';', error_bad_lines=False, encoding="latin-1")
    data_x = data.iloc[:, 0:4].values
    data_y = data.iloc[:, -1].values
    rmse_val = 0
    i = 0
    
    while (i < 10):
        imputer = KNNImputer(n_neighbors=10)
        data_x_knn_imputer = imputer.fit_transform(data_x)
        
        data_x_knn_imputer = pd.concat([pd.DataFrame(data_x_knn_imputer, columns = ["sepal_length", "sepal_width", "petal_length","petal_width"]), labels], axis=1)
        data_x_knn_imputer.to_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\KNN\Iris'+ str(item) + '_KNN.csv',  sep=';', encoding="latin-1", index = False)

        predict = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\KNN\Iris'+ str(item) + '_KNN.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
        real = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
 
        rmse_val  = rmse_val + rmse(np.array(predict), np.array(real))      
        i = i + 1
    print("Root Mean Square Error (" + str(item) + "% missing values): " + str(rmse_val/10))
    

MissForest Imputation

In [ ]:
from missingpy import MissForest

for item in miss_rate:
    data = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\DadosAusentes\Iris'+ str(item) + '.csv', sep=';', error_bad_lines=False, encoding="latin-1")
    data_x = data.iloc[:, 0:4].values
    data_y = data.iloc[:, -1].values
    rmse_val = 0
    i = 0
    
    while (i < 10):

        imputer = MissForest(max_iter=1)
        data_x_missforest_imputer = imputer.fit_transform(data_x)

        data_x_missforest_imputer = pd.concat([pd.DataFrame(data_x_missforest_imputer, columns = ["sepal_length", "sepal_width", "petal_length","petal_width"]), labels], axis=1)
        data_x_missforest_imputer.to_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\MissForest\Iris'+ str(item) + '_MissForest.csv',  sep=';', encoding="latin-1", index = False)

        predict = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\MissForest\Iris'+ str(item) + '_MissForest.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
        real = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
  
        rmse_val  = rmse_val + rmse(np.array(predict), np.array(real))      
        i = i + 1
    print("Root Mean Square Error (" + str(item) + "% missing values): " + str(rmse_val/10))
         

Multiple Imputation

In [ ]:
from imputena import mice

for item in miss_rate:
    data = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\DadosAusentes\Iris'+ str(item) + '.csv', sep=';', error_bad_lines=False, encoding="latin-1")
    data_x = data.iloc[:, 0:4].values
    data_x = pd.DataFrame(data_x, columns = ["sepal_length", "sepal_width", "petal_length","petal_width"])
    data_y = data.iloc[:, -1].values
    rmse_val = 0
    i = 0
    
    while (i < 10):
        data_x_mi_imputer = mice(data_x, imputations=5)

        zero_data = np.zeros(shape=(150,4))
        df = pd.DataFrame(zero_data, columns = ["sepal_length", "sepal_width", "petal_length","petal_width"])

        for d in data_x_mi_imputer:
            for col in d.columns:
                df[col]+=d[col]

        df = df/5

        data_x_mi_imputer = pd.concat([df, labels], axis=1)

        data_x_mi_imputer.to_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\MI\Iris'+ str(item) + '_MI.csv',  sep=';', encoding="latin-1", index = False)

        predict = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\MI\Iris'+ str(item) + '_MI.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
        real = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
  
        rmse_val  = rmse_val + rmse(np.array(predict), np.array(real))      
        i = i + 1
     
    print("Root Mean Square Error (" + str(item) + "% missing values): " + str(rmse_val/10))
    

KMeans Imputation

In [ ]:
import numpy as np
from sklearn.cluster import KMeans

def kmeans_missing(X, n_clusters, max_iter):
    """Perform K-Means clustering on data with missing values.

    Args:
      X: An [n_samples, n_features] array of data to cluster.
      n_clusters: Number of clusters to form.
      max_iter: Maximum number of EM iterations to perform.

    Returns:
      labels: An [n_samples] vector of integer labels.
      centroids: An [n_clusters, n_features] array of cluster centroids.
      X_hat: Copy of X with the missing values filled in.
    """

    # Initialize missing values to their column means
    missing = ~np.isfinite(X)
    mu = np.nanmean(X, 0, keepdims=1)
    X_hat = np.where(missing, mu, X)

    for i in range(max_iter):
        if i > 0:
            # initialize KMeans with the previous set of centroids. this is much
            # faster and makes it easier to check convergence (since labels
            # won't be permuted on every iteration), but might be more prone to
            # getting stuck in local minima.
            cls = KMeans(n_clusters, init=prev_centroids)
        else:
            # do multiple random initializations in parallel
            cls = KMeans(n_clusters, n_jobs=-1)

        # perform clustering on the filled-in data
        labels = cls.fit_predict(X_hat)
        centroids = cls.cluster_centers_

        # fill in the missing values based on their cluster centroids
        X_hat[missing] = centroids[labels][missing]

        # when the labels have stopped changing then we have converged
        if i > 0 and np.all(labels == prev_labels):
            break

        prev_labels = labels
        prev_centroids = cls.cluster_centers_

    return X_hat

for item in miss_rate:
    data = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\DadosAusentes\Iris'+ str(item) + '.csv', sep=';', error_bad_lines=False, encoding="latin-1")
    data_x = data.iloc[:, 0:4].values
    data_x = pd.DataFrame(data_x, columns = ["sepal_length", "sepal_width", "petal_length","petal_width"])
    data_y = data.iloc[:, -1].values
    rmse_val = 0
    i = 0
    
    while (i < 10):   
    
        data_x_kmeans_imputer = kmeans_missing(data_x, 3, 10)

        data_x_kmeans_imputer = pd.concat([pd.DataFrame(data_x_kmeans_imputer, columns = ["sepal_length", "sepal_width", "petal_length","petal_width"]), labels], axis=1)

        data_x_kmeans_imputer.to_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Kmeans\Iris'+ str(item) + '_Kmeans.csv',  sep=';', encoding="latin-1", index = False)

        predict = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Kmeans\Iris'+ str(item) + '_Kmeans.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
        real = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
  
        rmse_val  = rmse_val + rmse(np.array(predict), np.array(real))      
        i = i + 1
     
    print("Root Mean Square Error (" + str(item) + "% missing values): " + str(rmse_val/10))
    

Imputação por Regressão

In [ ]:
from imputena import linear_regression

for item in miss_rate:
    data = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\DadosAusentes\Iris'+ str(item) + '.csv', sep=';', error_bad_lines=False, encoding="latin-1")
    data_x = data.iloc[:, 0:4].values
    data_x = pd.DataFrame(data_x, columns = ["sepal_length", "sepal_width", "petal_length","petal_width"])
    data_y = data.iloc[:, -1].values
    rmse_val = 0
    i = 0
    
    while (i < 10):       
        data_x_lr_imputer = linear_regression(data_x, regressions='available')

        data_x_lr_imputer = pd.concat([data_x_lr_imputer, labels], axis=1)

        data_x_lr_imputer.to_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Regression\Iris'+ str(item) + '_RL.csv',  sep=';', encoding="latin-1", index = False)

        predict = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Regression\Iris'+ str(item) + '_RL.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))
        real = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))

        rmse_val  = rmse_val + rmse(np.array(predict), np.array(real))      
        i = i + 1
  
     
    print("Root Mean Square Error (" + str(item) + "% missing values): " + str(rmse_val/10))
    


GAIN Imputation

In [ ]:
#%% Packages
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn.functional as F
import warnings
warnings.filterwarnings("ignore")

import os
os.chdir("C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain")

#-------------------------------------------------------------------------
# Arquivo data_loader.py
#-------------------------------------------------------------------------
# Necessary packages
import numpy as np
import numpy
#from utils import binary_sampler
from keras.datasets import mnist
from python_utils import *
import pandas as pd

def binary_sampler(p, rows, cols):
    unif_random_matrix = np.random.uniform(0.0, 1.0, size=[rows, cols])
    binary_random_matrix = (unif_random_matrix < p).astype(np.float32)
    return binary_random_matrix


def data_loader (data_name, miss_rate):
  '''Loads datasets and introduce missingness.
  
  Args:
    - data_name: letter, spam, or mnist
    - miss_rate: the probability of missing components
    
  Returns:
    data_x: original data
    miss_data_x: data with missing values
    data_m: indicator matrix for missing components
  '''

  # Load data
  file_name = 'C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain\IrisGAIN.csv'  
  data_x = np.loadtxt(file_name, delimiter=",", skiprows=1)
 
  
  miss_data_x = np.genfromtxt(data_name, delimiter=";", skip_header=1)   
  data_m = miss_data_x.copy()
  
  coluna = 0
  linha = 0
  for x in data_m:
    
    for y in x:
      if np.isnan(y):
        data_m[linha,coluna]=0
        
      else:
        data_m[linha,coluna]=1
      coluna = coluna + 1
    linha = linha + 1
    coluna = 0
    
  a = numpy.asarray(data_x)
  b = numpy.asarray(miss_data_x)
  c = numpy.asarray(data_m)
  numpy.savetxt("C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\GAIN\data_x.csv", a, delimiter=",")
  numpy.savetxt("C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\GAIN\miss_data.csv", b, delimiter=",")
  numpy.savetxt("C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\GAIN\data_m.csv", c, delimiter=",")  

  return data_x, miss_data_x, data_m

#----------------------------------------------------------------------
# Arquivo utils.py
#----------------------------------------------------------------------
'''Utility functions for GAIN.
(1) normalization: MinMax Normalizer
(2) renormalization: Recover the data from normalzied data
(3) rounding: Handlecategorical variables after imputation
(4) rmse_loss: Evaluate imputed data in terms of RMSE
(5) xavier_init: Xavier initialization
(6) binary_sampler: sample binary random variables
(7) uniform_sampler: sample uniform random variables
(8) sample_batch_index: sample random batch index
'''
 
# Necessary packages
import numpy as np
#import tensorflow as tf
##IF USING TF 2 use following import to still use TF < 2.0 Functionalities
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()


def normalization (data, parameters=None):
  '''Normalize data in [0, 1] range.
  
  Args:
    - data: original data
  
  Returns:
    - norm_data: normalized data
    - norm_parameters: min_val, max_val for each feature for renormalization
  '''

  # Parameters
  _, dim = data.shape
  norm_data = data.copy()
  
  if parameters is None:
  
    # MixMax normalization
    min_val = np.zeros(dim)
    max_val = np.zeros(dim)
    
    # For each dimension
    for i in range(dim):
      min_val[i] = np.nanmin(norm_data[:,i])
      norm_data[:,i] = norm_data[:,i] - np.nanmin(norm_data[:,i])
      max_val[i] = np.nanmax(norm_data[:,i])
      norm_data[:,i] = norm_data[:,i] / (np.nanmax(norm_data[:,i]) + 1e-6)   
      
    # Return norm_parameters for renormalization
    norm_parameters = {'min_val': min_val,
                       'max_val': max_val}

  else:
    min_val = parameters['min_val']
    max_val = parameters['max_val']
    
    # For each dimension
    for i in range(dim):
      norm_data[:,i] = norm_data[:,i] - min_val[i]
      norm_data[:,i] = norm_data[:,i] / (max_val[i] + 1e-6)  
      
    norm_parameters = parameters    
      
  return norm_data, norm_parameters


def renormalization (norm_data, norm_parameters):
  '''Renormalize data from [0, 1] range to the original range.
  
  Args:
    - norm_data: normalized data
    - norm_parameters: min_val, max_val for each feature for renormalization
  
  Returns:
    - renorm_data: renormalized original data
  '''
  
  min_val = norm_parameters['min_val']
  max_val = norm_parameters['max_val']

  _, dim = norm_data.shape
  renorm_data = norm_data.copy()
    
  for i in range(dim):
    renorm_data[:,i] = renorm_data[:,i] * (max_val[i] + 1e-6)   
    renorm_data[:,i] = renorm_data[:,i] + min_val[i]
    
  return renorm_data


def rounding (imputed_data, data_x):
  '''Round imputed data for categorical variables.
  
  Args:
    - imputed_data: imputed data
    - data_x: original data with missing values
    
  Returns:
    - rounded_data: rounded imputed data
  '''
  
  _, dim = data_x.shape
  rounded_data = imputed_data.copy()
  
  for i in range(dim):
    temp = data_x[~np.isnan(data_x[:, i]), i]
    # Only for the categorical variable
    if len(np.unique(temp)) < 20:
      rounded_data[:, i] = np.round(rounded_data[:, i])
      
  return rounded_data


def rmse_loss (ori_data, imputed_data, data_m):
  '''Compute RMSE loss between ori_data and imputed_data
  
  Args:
    - ori_data: original data without missing values
    - imputed_data: imputed data
    - data_m: indicator matrix for missingness
    
  Returns:
    - rmse: Root Mean Squared Error
  '''
  
  ori_data, norm_parameters = normalization(ori_data)
  imputed_data, _ = normalization(imputed_data, norm_parameters)
    
  # Only for missing values
  nominator = np.sum(((1-data_m) * ori_data - (1-data_m) * imputed_data)**2)
  denominator = np.sum(1-data_m)
  
  rmse = np.sqrt(nominator/float(denominator))
  
  return rmse


def xavier_init(size):
  '''Xavier initialization.
  
  Args:
    - size: vector size
    
  Returns:
    - initialized random vector.
  '''
  in_dim = size[0]
  xavier_stddev = 1. / tf.sqrt(in_dim / 2.)
  return tf.random_normal(shape = size, stddev = xavier_stddev)
      

def binary_sampler(p, rows, cols):
  '''Sample binary random variables.
  
  Args:
    - p: probability of 1
    - rows: the number of rows
    - cols: the number of columns
    
  Returns:
    - binary_random_matrix: generated binary random matrix.
  '''
  unif_random_matrix = np.random.uniform(0., 1., size = [rows, cols])
  binary_random_matrix = 1*(unif_random_matrix < p)
  return binary_random_matrix


def uniform_sampler(low, high, rows, cols):
  '''Sample uniform random variables.
  
  Args:
    - low: low limit
    - high: high limit
    - rows: the number of rows
    - cols: the number of columns
    
  Returns:
    - uniform_random_matrix: generated uniform random matrix.
  '''
  return np.random.uniform(low, high, size = [rows, cols])       


def sample_batch_index(total, batch_size):
  '''Sample index of the mini-batch.
  
  Args:
    - total: total number of samples
    - batch_size: batch size
    
  Returns:
    - batch_idx: batch index
  '''
  total_idx = np.random.permutation(total)
  batch_idx = total_idx[:batch_size]
  return batch_idx

#---------------------------------------------------------------------
# Arquivo gain.py
#---------------------------------------------------------------------
'''GAIN function.
Date: 2020/02/28
Reference: J. Yoon, J. Jordon, M. van der Schaar, "GAIN: Missing Data 
           Imputation using Generative Adversarial Nets," ICML, 2018.
Paper Link: http://proceedings.mlr.press/v80/yoon18a/yoon18a.pdf
Contact: jsyoon0823@gmail.com
'''

# Necessary packages
#import tensorflow as tf
##IF USING TF 2 use following import to still use TF < 2.0 Functionalities
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()

import numpy as np
import numpy
from tqdm import tqdm




def gain (data_x, gain_parameters):
  '''Impute missing values in data_x
  
  Args:
    - data_x: original data with missing values
    - gain_parameters: GAIN network parameters:
      - batch_size: Batch size
      - hint_rate: Hint rate
      - alpha: Hyperparameter
      - iterations: Iterations
      
  Returns:
    - imputed_data: imputed data
  '''
  # Define mask matrix
  data_m = 1-np.isnan(data_x)
  
  # System parameters
  batch_size = gain_parameters['batch_size']
  hint_rate = gain_parameters['hint_rate']
  alpha = gain_parameters['alpha']
  iterations = gain_parameters['iterations']
  
  # Other parameters
  no, dim = data_x.shape
  
  # Hidden state dimensions
  h_dim = int(dim)
  
  # Normalization
  norm_data, norm_parameters = normalization(data_x)
  norm_data_x = np.nan_to_num(norm_data, 0)
  
  ## GAIN architecture   
  # Input placeholders
  # Data vector
  X = tf.placeholder(tf.float32, shape = [None, dim])
  # Mask vector 
  M = tf.placeholder(tf.float32, shape = [None, dim])
  # Hint vector
  H = tf.placeholder(tf.float32, shape = [None, dim])
  
  # Discriminator variables
  D_W1 = tf.Variable(xavier_init([dim*2, h_dim])) # Data + Hint as inputs
  D_b1 = tf.Variable(tf.zeros(shape = [h_dim]))
  
  D_W2 = tf.Variable(xavier_init([h_dim, h_dim]))
  D_b2 = tf.Variable(tf.zeros(shape = [h_dim]))
  
  D_W3 = tf.Variable(xavier_init([h_dim, dim]))
  D_b3 = tf.Variable(tf.zeros(shape = [dim]))  # Multi-variate outputs
  
  theta_D = [D_W1, D_W2, D_W3, D_b1, D_b2, D_b3]
  
  #Generator variables
  # Data + Mask as inputs (Random noise is in missing components)
  G_W1 = tf.Variable(xavier_init([dim*2, h_dim]))  
  G_b1 = tf.Variable(tf.zeros(shape = [h_dim]))
  
  G_W2 = tf.Variable(xavier_init([h_dim, h_dim]))
  G_b2 = tf.Variable(tf.zeros(shape = [h_dim]))
  
  G_W3 = tf.Variable(xavier_init([h_dim, dim]))
  G_b3 = tf.Variable(tf.zeros(shape = [dim]))
  
  theta_G = [G_W1, G_W2, G_W3, G_b1, G_b2, G_b3]
  
  ## GAIN functions
  # Generator
  def generator(x,m):
    # Concatenate Mask and Data
    inputs = tf.concat(values = [x, m], axis = 1) 
    G_h1 = tf.nn.relu(tf.matmul(inputs, G_W1) + G_b1)
    G_h2 = tf.nn.relu(tf.matmul(G_h1, G_W2) + G_b2)   
    # MinMax normalized output
    G_prob = tf.nn.sigmoid(tf.matmul(G_h2, G_W3) + G_b3) 
    return G_prob
      
  # Discriminator
  def discriminator(x, h):
    # Concatenate Data and Hint
    inputs = tf.concat(values = [x, h], axis = 1) 
    D_h1 = tf.nn.relu(tf.matmul(inputs, D_W1) + D_b1)  
    D_h2 = tf.nn.relu(tf.matmul(D_h1, D_W2) + D_b2)
    D_logit = tf.matmul(D_h2, D_W3) + D_b3
    D_prob = tf.nn.sigmoid(D_logit)
    return D_prob
  
  ## GAIN structure
  # Generator
  G_sample = generator(X, M)
 
  # Combine with observed data
  Hat_X = X * M + G_sample * (1-M)
  
  # Discriminator
  D_prob = discriminator(Hat_X, H)
  
  ## GAIN loss
  D_loss_temp = -tf.reduce_mean(M * tf.log(D_prob + 1e-8) \
                                + (1-M) * tf.log(1. - D_prob + 1e-8)) 
  
  G_loss_temp = -tf.reduce_mean((1-M) * tf.log(D_prob + 1e-8))
  
  MSE_loss = \
  tf.reduce_mean((M * X - M * G_sample)**2) / tf.reduce_mean(M)
  
  D_loss = D_loss_temp
  G_loss = G_loss_temp + alpha * MSE_loss 
  
  ## GAIN solver
  D_solver = tf.train.AdamOptimizer().minimize(D_loss, var_list=theta_D)
  G_solver = tf.train.AdamOptimizer().minimize(G_loss, var_list=theta_G)
  
  ## Iterations
  sess = tf.Session()
  sess.run(tf.global_variables_initializer())
   
  # Start Iterations
  for it in tqdm(range(iterations)):    
      
    # Sample batch
    batch_idx = sample_batch_index(no, batch_size)
    X_mb = norm_data_x[batch_idx, :]  
    M_mb = data_m[batch_idx, :]  
    # Sample random vectors  
    Z_mb = uniform_sampler(0, 0.01, batch_size, dim) 
    # Sample hint vectors
    H_mb_temp = binary_sampler(hint_rate, batch_size, dim)
    H_mb = M_mb * H_mb_temp
      
    # Combine random vectors with observed vectors
    X_mb = M_mb * X_mb + (1-M_mb) * Z_mb 
      
    _, D_loss_curr = sess.run([D_solver, D_loss_temp], 
                              feed_dict = {M: M_mb, X: X_mb, H: H_mb})
    _, G_loss_curr, MSE_loss_curr = \
    sess.run([G_solver, G_loss_temp, MSE_loss],
             feed_dict = {X: X_mb, M: M_mb, H: H_mb})
            
  ## Return imputed data      
  Z_mb = uniform_sampler(0, 0.01, no, dim) 
  M_mb = data_m
  X_mb = norm_data_x          
  X_mb = M_mb * X_mb + (1-M_mb) * Z_mb 
      
  imputed_data = sess.run([G_sample], feed_dict = {X: X_mb, M: M_mb})[0]
  
  imputed_data = data_m * norm_data_x + (1-data_m) * imputed_data
  
  # Renormalization
  imputed_data = renormalization(imputed_data, norm_parameters)  
  
  # Rounding
  imputed_data = rounding(imputed_data, data_x)  

  #a = numpy.asarray(imputed_data)
  #numpy.savetxt("imputed_data.csv", a, delimiter=",") 
          
  return imputed_data

#-------------------------------------------------------------
# Arquivo main.py
#-------------------------------------------------------------

# Necessary packages
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import argparse
import numpy as np

#from data_loader import data_loader
#from gain import gain
#from utils import rmse_loss

def rmse(predictions, targets):
    return np.sqrt(((predictions - targets) ** 2).mean())

def main (args):
  '''Main function for UCI letter and spam datasets.
  
  Args:
    - data_name: letter or spam
    - miss_rate: probability of missing components
    - batch:size: batch size
    - hint_rate: hint rate
    - alpha: hyperparameter
    - iterations: iterations
    
  Returns:
    - imputed_data_x: imputed data
    - rmse: Root Mean Squared Error
  '''
  
  data_name = args.data_name
  miss_rate = args.miss_rate
  
  gain_parameters = {'batch_size': args.batch_size,
                     'hint_rate': args.hint_rate,
                     'alpha': args.alpha,
                     'iterations': args.iterations}
  
  # Load data and introduce missingness
  #data_name = 'C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain\IrisGAIN.csv'
  miss_rate = [0.05,0.1,0.15,0.20,0.25,0.30,0.35,0.40,0.45,0.50]

  for item in miss_rate:
      
      data_name = 'C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain\Iris'+ str(item) + '.csv'
  
      ori_data_x, miss_data_x, data_m = data_loader(data_name, item)
    
      rmse_val = 0
      i = 0

      #Separação dos rótulos para a imputação     
      miss_data = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain\Iris'+ str(item) + '.csv', sep=';', error_bad_lines=False, encoding="latin-1")
      miss_data_x = miss_data.iloc[:, 0:4].values
      miss_data_y = miss_data.iloc[:, -1].values
      
      while (i < 10):
          imputed_data_x = gain(miss_data_x, gain_parameters)

          imputed_data_x = numpy.asarray(imputed_data_x)
          imputed_data_x = pd.DataFrame(imputed_data_x, columns = ["sepal_length", "sepal_width", "petal_length","petal_width"])
          imputed_data_x = pd.concat([imputed_data_x, labels], axis=1)
          imputed_data_x.to_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain\Iris'+ str(item) + '_Gain.csv',  sep=';', encoding="latin-1", index = False)

          imputed_data_x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain\Iris'+ str(item) + '_Gain.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))

          real = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1", usecols= ("sepal_length", "sepal_width", "petal_length","petal_width"))

          # Report the RMSE performance
          rmse_val  = rmse_val + rmse(np.array(imputed_data_x), np.array(real))      
          i = i + 1
       
      print("Root Mean Square Error (" + str(item) + "% missing values): " + str(rmse_val/10))
          #rmse_val = rmse(np.array(imputed_data_x), np.array(real))
      #print("Root Mean Square Error (" + str(item) + "% missing values): " + str(rmse_val))
      
      #print("Root Mean Square Error (" + str(item) + "% missing values): " + str(np.round(rmse_val, 4)))
      
        # Report the RMSE performance
      #rmse = rmse_loss (ori_data_x, imputed_data_x, data_m)

      #print()
      #print("Root Mean Square Error (" + str(item) + "% missing values): " + str(np.round(rmse, 4)))

      #return imputed_data_x, rmse    
        
if __name__ == '__main__':  
  
  # Inputs for the main function
  parser = argparse.ArgumentParser()
  parser.add_argument(
      '--data_name',
      choices=['IrisGAIN'],
      default='IrisGAIN',
      type=str)
  parser.add_argument(
      '--miss_rate',
      help='missing data probability',
      default=0.2,
      type=float)
  parser.add_argument(
      '--batch_size',
      help='the number of samples in mini-batch',
      default=128,
      type=int)
  parser.add_argument(
      '--hint_rate',
      help='hint probability',
      default=0.9,
      type=float)
  parser.add_argument(
      '--alpha',
      help='hyperparameter',
      default=100,
      type=float)
  parser.add_argument(
      '--iterations',
      help='number of training interations',
      default=1000,
      type=int)
  
  #args = parser.parse_args() 
  args = parser.parse_args(args=[])
  # Calls main function  
  #imputed_data, rmse = main(args)
  main(args)

=========================================================================

# Classifiers

# Tests with dataset imputed by mean

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")


#Pipeline estimator
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Cross-validation params
kf = KFold(n_splits=10)

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for item in miss_rate:
    #Load data
    
    x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Mean\Iris'+ str(item) + '_Mean.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
    y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Mean\Iris'+ str(item) + '_Mean.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
    
    #Dataset split
    from sklearn.model_selection import train_test_split
    x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
    print("-------------------------------------------------------------------------------------------")
    print("****Tests with dataset imputed by mean with " + str(item) + "% missing data.****" )
    print("-------------------------------------------------------------------------------------------")

    print("--------------------------------------------")
    print("****Cross-validation results****" )
    print("--------------------------------------------")

    
    fold = 1
    labels = ['setosa', 'versicolor','virginica']
    
    for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Decision Tree
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)

        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)
        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Decision Tree---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_AD/10,3))
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_RF/10,3))
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_MLP/10,3))
    
    
    
print('---------------------------------------------------------------')
    
    
    print('---------------------------------------------------------')
    print('--------Tests Results------------------------------------')
    print('---------------------------------------------------------')
        
    print('---------------------------------------------------------')
    print("Decision Tree Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeAD.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("Random Forest Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeRF.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("MLP Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeMLP.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    
    precision_valid_AD = [0,0,0]
    recall_valid_AD = [0,0,0]
    fscore_valid_AD = [0,0,0]

    precision_valid_RF = [0,0,0]
    recall_valid_RF = [0,0,0]
    fscore_valid_RF = [0,0,0]

    precision_valid_MLP = [0,0,0]
    recall_valid_MLP = [0,0,0]
    fscore_valid_MLP = [0,0,0]

# Tests with dataset imputed by KNN

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")


#Pipeline estimator
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Cross-validation params
kf = KFold(n_splits=10)

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for item in miss_rate:
    #Load data
    
    x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\KNN\Iris'+ str(item) + '_KNN.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
    y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\KNN\Iris'+ str(item) + '_KNN.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
    
    #Dataset split
    from sklearn.model_selection import train_test_split
    x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
    print("-------------------------------------------------------------------------------------------")
    print("****Tests with dataset imputed by KNN with " + str(item) + "% missing data.****" )
    print("-------------------------------------------------------------------------------------------")

    print("--------------------------------------------")
    print("****Cross-validation results****" )
    print("--------------------------------------------")

    
    fold = 1
    labels = ['setosa', 'versicolor','virginica']
    
    for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Decision Tree
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)

        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)
        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Decision Tree---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_AD/10,3))
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_RF/10,3))
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_MLP/10,3))
    
    
    
print('---------------------------------------------------------------')
    
    
    print('---------------------------------------------------------')
    print('--------Tests Results------------------------------------')
    print('---------------------------------------------------------')
        
    print('---------------------------------------------------------')
    print("Decision Tree Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeAD.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("Random Forest Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeRF.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("MLP Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeMLP.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    
    precision_valid_AD = [0,0,0]
    recall_valid_AD = [0,0,0]
    fscore_valid_AD = [0,0,0]

    precision_valid_RF = [0,0,0]
    recall_valid_RF = [0,0,0]
    fscore_valid_RF = [0,0,0]

    precision_valid_MLP = [0,0,0]
    recall_valid_MLP = [0,0,0]
    fscore_valid_MLP = [0,0,0]

# Tests with dataset imputed by MissForest

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")


#Pipeline estimator
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Cross-validation params
kf = KFold(n_splits=10)

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for item in miss_rate:
    #Load data
    
    x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\MissForest\Iris'+ str(item) + '_MissForest.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
    y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\MissForest\Iris'+ str(item) + '_MissForest.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
    
    #Dataset split
    from sklearn.model_selection import train_test_split
    x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
    print("-------------------------------------------------------------------------------------------")
    print("****Tests with dataset imputed by MissForest with " + str(item) + "% missing data.****" )
    print("-------------------------------------------------------------------------------------------")

    print("--------------------------------------------")
    print("****Cross-validation results****" )
    print("--------------------------------------------")

    
    fold = 1
    labels = ['setosa', 'versicolor','virginica']
    
    for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Decision Tree
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)

        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)
        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Decision Tree---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_AD/10,3))
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_RF/10,3))
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_MLP/10,3))
    
    
    
print('---------------------------------------------------------------')
    
    
    print('---------------------------------------------------------')
    print('--------Tests Results------------------------------------')
    print('---------------------------------------------------------')
        
    print('---------------------------------------------------------')
    print("Decision Tree Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeAD.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("Random Forest Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeRF.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("MLP Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeMLP.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    
    precision_valid_AD = [0,0,0]
    recall_valid_AD = [0,0,0]
    fscore_valid_AD = [0,0,0]

    precision_valid_RF = [0,0,0]
    recall_valid_RF = [0,0,0]
    fscore_valid_RF = [0,0,0]

    precision_valid_MLP = [0,0,0]
    recall_valid_MLP = [0,0,0]
    fscore_valid_MLP = [0,0,0]

# Tests with dataset imputed by Multiple Imputation

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")


#Pipeline estimator
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Cross-validation params
kf = KFold(n_splits=10)

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for item in miss_rate:
    #Load data
    
    x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\MI\Iris'+ str(item) + '_MI.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
    y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\MI\Iris'+ str(item) + '_MI.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
    
    #Dataset split
    from sklearn.model_selection import train_test_split
    x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
    print("-------------------------------------------------------------------------------------------")
    print("****Tests with dataset imputed by MI with " + str(item) + "% missing data.****" )
    print("-------------------------------------------------------------------------------------------")

    print("--------------------------------------------")
    print("****Cross-validation results****" )
    print("--------------------------------------------")

    
    fold = 1
    labels = ['setosa', 'versicolor','virginica']
    
    for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Decision Tree
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)

        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)
        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Decision Tree---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_AD/10,3))
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_RF/10,3))
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_MLP/10,3))
    
    
    
print('---------------------------------------------------------------')
    
    
    print('---------------------------------------------------------')
    print('--------Tests Results------------------------------------')
    print('---------------------------------------------------------')
        
    print('---------------------------------------------------------')
    print("Decision Tree Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeAD.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("Random Forest Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeRF.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("MLP Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeMLP.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    
    precision_valid_AD = [0,0,0]
    recall_valid_AD = [0,0,0]
    fscore_valid_AD = [0,0,0]

    precision_valid_RF = [0,0,0]
    recall_valid_RF = [0,0,0]
    fscore_valid_RF = [0,0,0]

    precision_valid_MLP = [0,0,0]
    recall_valid_MLP = [0,0,0]
    fscore_valid_MLP = [0,0,0]

# Tests with dataset imputed by Clustering

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")


#Pipeline estimator
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Cross-validation params
kf = KFold(n_splits=10)

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for item in miss_rate:
    #Load data
    
    x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\KMeans\Iris'+ str(item) + '_KMeans.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
    y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\KMeans\Iris'+ str(item) + '_KMeans.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
    
    #Dataset split
    from sklearn.model_selection import train_test_split
    x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
    print("-------------------------------------------------------------------------------------------")
    print("****Tests with dataset imputed by KMeans with " + str(item) + "% missing data.****" )
    print("-------------------------------------------------------------------------------------------")

    print("--------------------------------------------")
    print("****Cross-validation results****" )
    print("--------------------------------------------")

    
    fold = 1
    labels = ['setosa', 'versicolor','virginica']
    
    for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Decision Tree
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)

        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)
        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Decision Tree---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_AD/10,3))
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_RF/10,3))
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_MLP/10,3))
    
    
    
print('---------------------------------------------------------------')
    
    
    print('---------------------------------------------------------')
    print('--------Tests Results------------------------------------')
    print('---------------------------------------------------------')
        
    print('---------------------------------------------------------')
    print("Decision Tree Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeAD.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("Random Forest Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeRF.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("MLP Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeMLP.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    
    precision_valid_AD = [0,0,0]
    recall_valid_AD = [0,0,0]
    fscore_valid_AD = [0,0,0]

    precision_valid_RF = [0,0,0]
    recall_valid_RF = [0,0,0]
    fscore_valid_RF = [0,0,0]

    precision_valid_MLP = [0,0,0]
    recall_valid_MLP = [0,0,0]
    fscore_valid_MLP = [0,0,0]

# Tests with dataset imputed by Regression

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")


#Pipeline estimator
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Cross-validation params
kf = KFold(n_splits=10)

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for item in miss_rate:
    #Load data
    
    x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Regression\Iris'+ str(item) + '_RL.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
    y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Regression\Iris'+ str(item) + '_RL.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
    
    #Dataset split
    from sklearn.model_selection import train_test_split
    x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
    print("-------------------------------------------------------------------------------------------")
    print("****Tests with dataset imputed by Regression with " + str(item) + "% missing data.****" )
    print("-------------------------------------------------------------------------------------------")

    print("--------------------------------------------")
    print("****Cross-validation results****" )
    print("--------------------------------------------")

    
    fold = 1
    labels = ['setosa', 'versicolor','virginica']
    
    for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Decision Tree
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)

        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)
        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Decision Tree---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_AD/10,3))
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_RF/10,3))
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_MLP/10,3))
    
    
    
print('---------------------------------------------------------------')
    
    
    print('---------------------------------------------------------')
    print('--------Tests Results------------------------------------')
    print('---------------------------------------------------------')
        
    print('---------------------------------------------------------')
    print("Decision Tree Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeAD.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("Random Forest Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeRF.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("MLP Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeMLP.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    
    precision_valid_AD = [0,0,0]
    recall_valid_AD = [0,0,0]
    fscore_valid_AD = [0,0,0]

    precision_valid_RF = [0,0,0]
    recall_valid_RF = [0,0,0]
    fscore_valid_RF = [0,0,0]

    precision_valid_MLP = [0,0,0]
    recall_valid_MLP = [0,0,0]
    fscore_valid_MLP = [0,0,0]

# Tests with dataset imputed by GAIN

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")


#Pipeline estimator
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Cross-validation params
kf = KFold(n_splits=10)

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for item in miss_rate:
    #Load data
    
    x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain\Iris'+ str(item) + '_Gain.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
    y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Gain\Iris'+ str(item) + '_Gain.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
    
    #Dataset split
    from sklearn.model_selection import train_test_split
    x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
    print("-------------------------------------------------------------------------------------------")
    print("****Tests with dataset imputed by Gain with " + str(item) + "% missing data.****" )
    print("-------------------------------------------------------------------------------------------")

    print("--------------------------------------------")
    print("****Cross-validation results****" )
    print("--------------------------------------------")

    
    fold = 1
    labels = ['setosa', 'versicolor','virginica']
    
    for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Decision Tree
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)

        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)
        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Decision Tree---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_AD/10,3))
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_RF/10,3))
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_MLP/10,3))
    
    
    
print('---------------------------------------------------------------')
    
    
    print('---------------------------------------------------------')
    print('--------Tests Results------------------------------------')
    print('---------------------------------------------------------')
        
    print('---------------------------------------------------------')
    print("Decision Tree Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeAD.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("Random Forest Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeRF.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("MLP Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeMLP.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    
    precision_valid_AD = [0,0,0]
    recall_valid_AD = [0,0,0]
    fscore_valid_AD = [0,0,0]

    precision_valid_RF = [0,0,0]
    recall_valid_RF = [0,0,0]
    fscore_valid_RF = [0,0,0]

    precision_valid_MLP = [0,0,0]
    recall_valid_MLP = [0,0,0]
    fscore_valid_MLP = [0,0,0]

# Tests with dataset imputed by Hot-deck

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")


#Pipeline estimator
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Cross-validation params
kf = KFold(n_splits=10)

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for item in miss_rate:
    #Load data
    
    x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Hotdeck\Iris'+ str(item) + '_HD.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
    y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Hotdeck\Iris'+ str(item) + '_HD.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
    
    #Dataset split
    from sklearn.model_selection import train_test_split
    x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
    print("-------------------------------------------------------------------------------------------")
    print("****Tests with dataset imputed by Hotdeck with " + str(item) + "% missing data.****" )
    print("-------------------------------------------------------------------------------------------")

    print("--------------------------------------------")
    print("****Cross-validation results****" )
    print("--------------------------------------------")

    
    fold = 1
    labels = ['setosa', 'versicolor','virginica']
    
    for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Decision Tree
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)

        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)
        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        
        #Computing metrics
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None,labels=['setosa', 'versicolor', 'virginica'])        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Decision Tree---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_AD/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_AD/10,3))
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_RF/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_RF/10,3))
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
print("Average Precision (setosa, versicolor and virginica classes): " , np.round(precision_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average Recall (setosa, versicolor and virginica classes): " ,  np.round(recall_valid_MLP/10,3))
print("----------------------------------------------------------------")
print("Average F1 score (setosa, versicolor and virginica classes): " ,  np.round(fscore_valid_MLP/10,3))
    
    
    
print('---------------------------------------------------------------')
    
    
    print('---------------------------------------------------------')
    print('--------Tests Results------------------------------------')
    print('---------------------------------------------------------')
        
    print('---------------------------------------------------------')
    print("Decision Tree Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeAD.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("Random Forest Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeRF.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    print("MLP Classfication Report")
    print('---------------------------------------------------------')
    predito_teste = pipeMLP.predict(x_teste)
    print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

    print('---------------------------------------------------------')
    
    precision_valid_AD = [0,0,0]
    recall_valid_AD = [0,0,0]
    fscore_valid_AD = [0,0,0]

    precision_valid_RF = [0,0,0]
    recall_valid_RF = [0,0,0]
    fscore_valid_RF = [0,0,0]

    precision_valid_MLP = [0,0,0]
    recall_valid_MLP = [0,0,0]
    fscore_valid_MLP = [0,0,0]

# Testes sem dados ausentes

In [ ]:
#Código de testes com a base imputada com Regressão Linear

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
import sklearn.metrics as metrics
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support
import numpy as np
import warnings
warnings.filterwarnings("ignore")

#Pipeline estimador
pipeAD = Pipeline([['scaler', StandardScaler()],
                 ['classifier', DecisionTreeClassifier(criterion='entropy', max_depth=10, random_state=2021)]])

pipeRF = Pipeline([['scaler', StandardScaler()],
                 ['classifier', RandomForestClassifier(criterion='entropy', n_estimators=100, max_depth=10,random_state=2021)]])

pipeMLP = Pipeline([['scaler', StandardScaler()],
                 ['classifier', MLPClassifier(max_iter=100, learning_rate_init=0.2, momentum=0.3, hidden_layer_sizes=15, random_state=2021)]])

#Definindo o número de folds para a validação cruzada
kf = KFold(n_splits=10)

#Testes com a imputação Regressão Linear

#Load data
    
x = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['sepal_length', 'sepal_width', 'petal_length','petal_width']))
y = pd.read_csv('C:\Anaconda3\Scripts\TestesPesquisa\Bases\Iris\Original\Iris.csv', sep=';', error_bad_lines=False, encoding="latin-1",  usecols= (['species']))
     
#SEPARANDO DATASET DE TESTE
from sklearn.model_selection import train_test_split
x, x_teste, y, y_teste = train_test_split(x, y, test_size=0.1, stratify=y, random_state=2021)
    
print("--------------------------------------------")
print("Resultados da Validação Cruzada" )
print("--------------------------------------------")

    
fold = 1

precision_valid_AD = [0,0,0]
recall_valid_AD = [0,0,0]
fscore_valid_AD = [0,0,0]

precision_valid_RF = [0,0,0]
recall_valid_RF = [0,0,0]
fscore_valid_RF = [0,0,0]

precision_valid_MLP = [0,0,0]
recall_valid_MLP = [0,0,0]
fscore_valid_MLP = [0,0,0]

for train_index, valid_index in kf.split(x):
        x_train = x.iloc[train_index].loc[:]
        y_train = y.iloc[train_index]

        x_valid = x.iloc[valid_index].loc[:]
        y_valid = y.iloc[valid_index]

        #Árvore de decisão
        pipeAD.fit(x_train, y_train)
        predito = pipeAD.predict(x_valid)
        #print('Árvore de Decisão FOLD: ' + str(fold))
        #print(metrics.classification_report(y_valid,predito, target_names=labels))

        #cm = metrics.confusion_matrix(y_valid,predito, labels=labels)
        #disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        #disp.plot()
        #plt.show()

        #Calculando as métricas
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None)        
        precision_valid_AD = np.add(precision_valid_AD,precision)
        recall_valid_AD = np.add(recall_valid_AD,recall)
        fscore_valid_AD = np.add(fscore_valid_AD,fscore)     
       
        #Random Forest
        pipeRF.fit(x_train, y_train)
        predito = pipeRF.predict(x_valid)
        #print('Random Forest FOLD: ' + str(fold))
        #print(metrics.classification_report(y_valid,predito, target_names=labels))

        #cm = metrics.confusion_matrix(y_valid,predito, labels=labels)
        #disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        #disp.plot()
        #plt.show()
        
        #Calculando as métricas
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None)        
        precision_valid_RF = np.add(precision_valid_RF,precision)
        recall_valid_RF = np.add(recall_valid_RF,recall)
        fscore_valid_RF = np.add(fscore_valid_RF,fscore)

        
        #MLP
        pipeMLP.fit(x_train, y_train)
        predito = pipeMLP.predict(x_valid)
        #print('MLP FOLD: ' + str(fold))
        #print(metrics.classification_report(y_valid,predito, target_names=labels))

        #cm = metrics.confusion_matrix(y_valid,predito, labels=labels)
        #disp = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        #disp.plot()
        #plt.show()
        
        #Calculando as métricas
        precision,recall,fscore,support = precision_recall_fscore_support(y_valid, predito, average=None)        
        precision_valid_MLP = np.add(precision_valid_MLP,precision)
        recall_valid_MLP = np.add(recall_valid_MLP,recall)
        fscore_valid_MLP = np.add(fscore_valid_MLP,fscore)


        fold += 1

print('-----------------Árvore de Decisão---------------------------------')
#print("Precisão média na validação das classes setosa, versicolor e virginica: " , sum(np.round(precision_valid_AD/10,2))/3)
print("----------------------------------------------------------------")
#print("Recall médio na validação das classes setosa, versicolor e virginica: " ,  sum(np.round(recall_valid_AD/10,2))/3)
#print("----------------------------------------------------------------")
print("F1 score médio na validação das classes setosa, versicolor e virginica: " ,  sum(np.round(fscore_valid_AD/10,2))/3)
    
print('---------------------------------------------------------------')

print('-----------------Random Forest---------------------------------')
#print("Precisão média na validação das classes setosa, versicolor e virginica: " , sum(np.round(precision_valid_RF/10,2))/3)
#print("----------------------------------------------------------------")
#print("Recall médio na validação das classes setosa, versicolor e virginica: " , sum(np.round(recall_valid_RF/10,2))/3)
print("----------------------------------------------------------------")
print("F1 score médio na validação das classes setosa, versicolor e virginica: " , sum(np.round(fscore_valid_RF/10,2))/3) 
    
print('---------------------------------------------------------------')

print('----------------------------MLP---------------------------------')
#print("Precisão média na validação das classes setosa, versicolor e virginica: " , sum(np.round(precision_valid_MLP/10,2))/3)
#print("----------------------------------------------------------------")
#print("Recall médio na validação das classes setosa, versicolor e virginica: " , sum(np.round(recall_valid_MLP/10,2))/3)
print("----------------------------------------------------------------")
print("F1 score médio na validação das classes setosa, versicolor e virginica: " ,sum(np.round(fscore_valid_MLP/10,2))/3) 
    
    
print('---------------------------------------------------------------')
    


In [ ]:
print('---------------------------------------------------------')
print("Acurácia nos Testes com AD")
print('---------------------------------------------------------')
print(pipeAD.score(x_teste, y_teste))

print('---------------------------------------------------------')
print("Relatório de Classificação dos Testes com AD ")
print('---------------------------------------------------------')
predito_teste = pipeAD.predict(x_teste)
print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

print('---------------------------------------------------------')
print("Acurácia nos testes com Random Forest")
print('---------------------------------------------------------')
print(pipeRF.score(x_teste, y_teste))

print('---------------------------------------------------------')
print("Relatório de Classificação dos Testes com Random Forest ")
print('---------------------------------------------------------')
predito_teste = pipeRF.predict(x_teste)
print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 

print('---------------------------------------------------------')
print("Acurácia nos testes com MLP")
print('---------------------------------------------------------')
print(pipeMLP.score(x_teste, y_teste))

print('---------------------------------------------------------')
print("Relatório de Classificação dos Testes com MLP ")
print('---------------------------------------------------------')
predito_teste = pipeMLP.predict(x_teste)
print(metrics.classification_report(y_teste,predito_teste, target_names=['setosa', 'versicolor','virginica'])) 